<a href="https://colab.research.google.com/github/edwardoughton/IGARSS26/blob/main/notebook_1_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛰️ IGARSS 2026 Summer School — Part 1 of 3 🛰️
## AI Refusnik to AI Evangelist: Excellent at AI-Assisted Coding for Satellite Image Analysis

**Instructor:** Prof. Edward Oughton, George Mason University

**Session duration:** ~60 minutes

---

Welcome to Part 1 of this three-part summer school session!

In this first notebook we will cover how to **programmatically download and process Landsat and Sentinel-2 satellite imagery** using Python and cloud-based STAC catalogs.

By the end of Part 1, you will be able to:
- Understand what STAC is and why it matters for large-scale EO workflows
- Query and download Landsat Collection 2 imagery via Microsoft Planetary Computer
- Query and download Sentinel-2 imagery via the same API
- Compute and visualize key spectral indices (NDVI, NDWI, NDBI)
- Save processed imagery as analysis-ready GeoTIFFs

> **Audience:** First/second year PhD students with some Python experience but limited satellite image processing background.

## 📗 Learning Objectives 📗

By the end of this notebook, you should be able to:

1. Explain the STAC (SpatioTemporal Asset Catalog) standard and how it enables cloud-native EO workflows
2. Search and retrieve Landsat 8/9 (Collection 2, Level 2) scenes via `pystac-client`
3. Search and retrieve Sentinel-2 Level 2A scenes via the same API
4. Load and inspect raster metadata (CRS, resolution, band structure)
5. Compute spectral indices: NDVI, NDWI, NDBI
6. Visualize true-colour and false-colour composites
7. Save analysis-ready outputs as GeoTIFF files

---

## 0. Background: Why STAC?

Traditionally, downloading satellite imagery required navigating agency portals (USGS EarthExplorer, ESA Copernicus Open Access Hub), manually selecting scenes, and waiting for large downloads.

**STAC (SpatioTemporal Asset Catalog)** is an open standard that describes geospatial datasets as machine-readable JSON, enabling:

- Programmatic search by bounding box, date range, cloud cover, collection
- Cloud-native access — stream only the pixels you need, no full-scene download required
- Consistent API across providers (Microsoft Planetary Computer, AWS Earth on Demand, Element84 etc.)

We will use **Microsoft Planetary Computer** throughout this session as it hosts both Landsat Collection 2 and Sentinel-2 Level 2A with free, signed access.

Key Python libraries:

| Library | Role |
|---|---|
| `pystac-client` | Search STAC catalogs |
| `planetary-computer` | Sign Planetary Computer asset URLs |
| `odc-stac` | Load STAC assets as xarray Datasets |
| `rasterio` | Read/write raster files, affine transforms |
| `numpy` | Numerical array operations |
| `matplotlib` | Visualization |

## 1. Install and import dependencies

In [ ]:
# Install required packages (run once in Colab)
!pip -q install pystac-client planetary-computer odc-stac rasterio numpy matplotlib geopandas shapely requests

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.plot import show
from rasterio.transform import from_origin
from rasterio.crs import CRS

import pystac_client
import planetary_computer
import odc.stac

warnings.filterwarnings('ignore')

# Create output directory
DATA_DIR = Path('igarss26_data')
DATA_DIR.mkdir(exist_ok=True)

print('All packages imported successfully.')
print(f'Output directory: {DATA_DIR.resolve()}')

---

## 2. Connect to Microsoft Planetary Computer STAC API

The Planetary Computer catalog hosts petabytes of curated Earth observation data. We connect to it using `pystac_client.Client.open()` and pass a `modifier` that automatically signs asset URLs so we can stream pixel data.

In [ ]:
STAC_API_URL = 'https://planetarycomputer.microsoft.com/api/stac/v1'

catalog = pystac_client.Client.open(
    STAC_API_URL,
    modifier=planetary_computer.sign_inplace
)

print(f'Connected to: {catalog.title}')
print('Available collections (first 10):')
for i, col in enumerate(catalog.get_collections()):
    print(f'  {col.id}')
    if i >= 9:
        break

---

## 3. Define a study area and search parameters

We define our area of interest (AOI) as a bounding box in EPSG:4326 (WGS84 lon/lat). We will use **Washington D.C. and the surrounding Potomac River corridor** as a demonstration area — a region with interesting urban, water, and vegetation gradients.

In [ ]:
# Bounding box: [min_lon, min_lat, max_lon, max_lat]
# Washington D.C. metro area
AOI_BBOX = [-77.20, 38.75, -76.90, 39.05]

# Time window of interest
TIME_RANGE = '2024-06-01/2024-09-30'

# Maximum cloud cover percentage
MAX_CLOUD = 15

print(f'Study area bounding box: {AOI_BBOX}')
print(f'Time range: {TIME_RANGE}')
print(f'Max cloud cover: {MAX_CLOUD}%')

---

## 4. Download and Process Landsat Collection 2 Level 2

**Landsat Collection 2 Level 2** provides atmospherically corrected surface reflectance products. This is the recommended starting point for most land surface analysis.

### Landsat band structure (relevant bands)

| Band | Name | Wavelength (µm) | Key uses |
|---|---|---|---|
| B2 | Blue | 0.45–0.51 | Water depth, coastal |
| B3 | Green | 0.53–0.59 | Vegetation health |
| B4 | Red | 0.64–0.67 | Vegetation, soil |
| B5 | NIR | 0.85–0.88 | Biomass, NDVI |
| B6 | SWIR-1 | 1.57–1.65 | Soil moisture, NDWI |
| B7 | SWIR-2 | 2.11–2.29 | Geology, minerals |

Spatial resolution: **30 m** (multispectral)

In [ ]:
# Search for Landsat Collection 2 Level 2 scenes
landsat_search = catalog.search(
    collections=['landsat-c2-l2'],
    bbox=AOI_BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': MAX_CLOUD}},
)

landsat_items = list(landsat_search.items())
print(f'Found {len(landsat_items)} Landsat scenes')

if landsat_items:
    # Show metadata for the first (least cloudy) item
    best_item = sorted(landsat_items, key=lambda x: x.properties.get('eo:cloud_cover', 100))[0]
    print(f'\nBest scene:')
    print(f'  ID: {best_item.id}')
    print(f'  Date: {best_item.datetime}')
    print(f'  Cloud cover: {best_item.properties.get("eo:cloud_cover", "N/A")}%')
    print(f'  Platform: {best_item.properties.get("platform", "N/A")}')
    print(f'  Available assets: {list(best_item.assets.keys())[:10]}')

In [ ]:
# Load Landsat RGB + NIR + SWIR bands via odc-stac
# odc-stac loads data as an xarray Dataset, automatically handling
# CRS reprojection and resampling.

LANDSAT_BANDS = ['red', 'green', 'blue', 'nir08', 'swir16', 'swir22']

# Load the best scene clipped to our AOI
landsat_ds = odc.stac.load(
    [best_item],
    bands=LANDSAT_BANDS,
    bbox=AOI_BBOX,
    resolution=30,
    groupby='solar_day'
)

print('Landsat dataset loaded:')
print(landsat_ds)

In [ ]:
# Extract the first time slice as a numpy array
# Surface reflectance values are scaled: divide by 10000 to get 0-1 range
ls_scene = landsat_ds.isel(time=0)

red   = ls_scene['red'].values.astype(float)   / 10000.0
green = ls_scene['green'].values.astype(float) / 10000.0
blue  = ls_scene['blue'].values.astype(float)  / 10000.0
nir   = ls_scene['nir08'].values.astype(float) / 10000.0
swir1 = ls_scene['swir16'].values.astype(float)/ 10000.0

# Clip to valid reflectance range [0, 1]
def clip_reflectance(band):
    return np.clip(band, 0, 1)

red, green, blue, nir, swir1 = [
    clip_reflectance(b) for b in [red, green, blue, nir, swir1]
]

print(f'Image dimensions: {red.shape} (rows x cols)')
print(f'Red band range: [{red.min():.3f}, {red.max():.3f}]')
print(f'NIR band range: [{nir.min():.3f}, {nir.max():.3f}]')

In [ ]:
# Visualize the Landsat true-colour composite
def stretch(arr, low=2, high=98):
    """Apply percentile stretch for better visualization."""
    lo, hi = np.nanpercentile(arr, [low, high])
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)

rgb = np.dstack([stretch(red), stretch(green), stretch(blue)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(rgb)
axes[0].set_title('Landsat True-Colour (RGB)', fontsize=13)
axes[0].axis('off')

# False-colour composite: NIR-Red-Green highlights vegetation in red
nrg = np.dstack([stretch(nir), stretch(red), stretch(green)])
axes[1].imshow(nrg)
axes[1].set_title('Landsat False-Colour (NIR-R-G)\nVegetation appears red', fontsize=13)
axes[1].axis('off')

plt.suptitle('Landsat Collection 2 Level 2 — Washington D.C. area', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 4.1 Compute Spectral Indices from Landsat

Spectral indices are mathematical combinations of reflectance bands that enhance specific surface features:

**NDVI (Normalized Difference Vegetation Index)**
$$\text{NDVI} = \frac{\text{NIR} - \text{Red}}{\text{NIR} + \text{Red}}$$
Range: -1 to 1. Dense vegetation: 0.6–0.9. Bare soil: 0.1–0.2. Water: negative.

**NDWI (Normalized Difference Water Index)**
$$\text{NDWI} = \frac{\text{Green} - \text{NIR}}{\text{Green} + \text{NIR}}$$
Range: -1 to 1. Water bodies: positive values.

**NDBI (Normalized Difference Built-up Index)**
$$\text{NDBI} = \frac{\text{SWIR1} - \text{NIR}}{\text{SWIR1} + \text{NIR}}$$
Range: -1 to 1. Built-up areas: positive values.

In [ ]:
def safe_index(a, b):
    """Compute normalized difference index, avoiding division by zero."""
    denominator = a + b
    return np.where(denominator == 0, np.nan, (a - b) / denominator)

ndvi_ls = safe_index(nir, red)
ndwi_ls = safe_index(green, nir)
ndbi_ls = safe_index(swir1, nir)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

im0 = axes[0].imshow(ndvi_ls, cmap='RdYlGn', vmin=-0.3, vmax=0.8)
axes[0].set_title('NDVI\n(vegetation index)', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndwi_ls, cmap='Blues', vmin=-0.5, vmax=0.5)
axes[1].set_title('NDWI\n(water index)', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(ndbi_ls, cmap='YlOrRd', vmin=-0.5, vmax=0.5)
axes[2].set_title('NDBI\n(built-up index)', fontsize=12)
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.suptitle('Landsat Spectral Indices', fontsize=14)
plt.tight_layout()
plt.show()

print(f'NDVI  — mean: {np.nanmean(ndvi_ls):.3f}, std: {np.nanstd(ndvi_ls):.3f}')
print(f'NDWI  — mean: {np.nanmean(ndwi_ls):.3f}, std: {np.nanstd(ndwi_ls):.3f}')
print(f'NDBI  — mean: {np.nanmean(ndbi_ls):.3f}, std: {np.nanstd(ndbi_ls):.3f}')

In [ ]:
# Save Landsat NDVI as a GeoTIFF for downstream use
# We read the georeferencing from the odc-stac geobox

def save_single_band_geotiff(array, geobox, out_path, nodata=np.nan):
    """
    Save a 2D numpy array as a single-band GeoTIFF,
    preserving the spatial reference from an odc-stac geobox.
    """
    transform = geobox.transform
    crs = geobox.crs.to_wkt()

    with rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=array.shape[0],
        width=array.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs=crs,
        transform=transform,
        nodata=nodata,
        compress='lzw'
    ) as dst:
        dst.write(array.astype(np.float32), 1)

    print(f'Saved: {out_path}')

geobox = ls_scene.odc.geobox

save_single_band_geotiff(ndvi_ls,  geobox, DATA_DIR / 'landsat_ndvi.tif')
save_single_band_geotiff(ndwi_ls,  geobox, DATA_DIR / 'landsat_ndwi.tif')
save_single_band_geotiff(ndbi_ls,  geobox, DATA_DIR / 'landsat_ndbi.tif')

### ✏️ Exercise 4.1

The **EVI (Enhanced Vegetation Index)** was developed to improve sensitivity in high-biomass areas and reduce atmospheric and soil noise. It is calculated as:

$$\text{EVI} = 2.5 \times \frac{\text{NIR} - \text{Red}}{\text{NIR} + 6 \times \text{Red} - 7.5 \times \text{Blue} + 1}$$

1. Implement the EVI calculation using the Landsat bands loaded above.
2. Clip the result to the valid range [-1, 1].
3. Plot EVI side-by-side with NDVI. Where are the biggest differences?
4. Save your EVI result as `landsat_evi.tif`.

*Hint: Divide by zero situations can be handled with `np.where(denominator == 0, np.nan, ...)`*

In [ ]:
# Your code here


---

## 5. Download and Process Sentinel-2 Level 2A

**Sentinel-2** is a twin-satellite ESA mission providing multispectral imagery at **10 m, 20 m, and 60 m** spatial resolution — significantly finer than Landsat's 30 m.

### Sentinel-2 band structure (key bands)

| Band | Name | Wavelength (µm) | Resolution | Key uses |
|---|---|---|---|---|
| B02 | Blue | 0.490 | 10 m | True colour |
| B03 | Green | 0.560 | 10 m | True colour |
| B04 | Red | 0.665 | 10 m | Vegetation, NDVI |
| B08 | NIR | 0.842 | 10 m | Biomass, NDVI |
| B8A | Narrow NIR | 0.865 | 20 m | Red-edge analysis |
| B11 | SWIR-1 | 1.610 | 20 m | Soil moisture |
| B12 | SWIR-2 | 2.190 | 20 m | Geology |
| B05 | Red Edge 1 | 0.705 | 20 m | Chlorophyll |
| B06 | Red Edge 2 | 0.740 | 20 m | Canopy health |

Level 2A = **Bottom-of-Atmosphere (BOA) surface reflectance**, atmospherically corrected.

In [ ]:
# Search for Sentinel-2 Level 2A scenes
s2_search = catalog.search(
    collections=['sentinel-2-l2a'],
    bbox=AOI_BBOX,
    datetime=TIME_RANGE,
    query={'eo:cloud_cover': {'lt': MAX_CLOUD}},
)

s2_items = list(s2_search.items())
print(f'Found {len(s2_items)} Sentinel-2 scenes')

if s2_items:
    best_s2 = sorted(s2_items, key=lambda x: x.properties.get('eo:cloud_cover', 100))[0]
    print(f'\nBest scene:')
    print(f'  ID: {best_s2.id}')
    print(f'  Date: {best_s2.datetime}')
    print(f'  Cloud cover: {best_s2.properties.get("eo:cloud_cover", "N/A")}%')
    print(f'  Tile: {best_s2.properties.get("s2:mgrs_tile", "N/A")}')
    print(f'  Available assets: {list(best_s2.assets.keys())}')

In [ ]:
# Load Sentinel-2 bands at 10 m resolution
# We request all 10 m native bands plus resampled SWIR at 10 m
S2_BANDS = ['B02', 'B03', 'B04', 'B08']  # 10 m bands: Blue, Green, Red, NIR

s2_ds = odc.stac.load(
    [best_s2],
    bands=S2_BANDS,
    bbox=AOI_BBOX,
    resolution=10,
    groupby='solar_day'
)

print('Sentinel-2 dataset loaded:')
print(s2_ds)

In [ ]:
# Extract bands — Sentinel-2 L2A values are scaled by 10000
s2_scene = s2_ds.isel(time=0)

s2_blue  = clip_reflectance(s2_scene['B02'].values.astype(float) / 10000.0)
s2_green = clip_reflectance(s2_scene['B03'].values.astype(float) / 10000.0)
s2_red   = clip_reflectance(s2_scene['B04'].values.astype(float) / 10000.0)
s2_nir   = clip_reflectance(s2_scene['B08'].values.astype(float) / 10000.0)

print(f'Image shape: {s2_red.shape}')
print(f'At 10 m resolution, {s2_red.shape[0]*s2_red.shape[1]:,} pixels total')

In [ ]:
# Visualize Sentinel-2 true-colour image
s2_rgb = np.dstack([stretch(s2_red), stretch(s2_green), stretch(s2_blue)])
s2_nrg = np.dstack([stretch(s2_nir), stretch(s2_red), stretch(s2_green)])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].imshow(s2_rgb)
axes[0].set_title('Sentinel-2 True-Colour (RGB) — 10 m', fontsize=13)
axes[0].axis('off')

axes[1].imshow(s2_nrg)
axes[1].set_title('Sentinel-2 False-Colour (NIR-R-G) — 10 m', fontsize=13)
axes[1].axis('off')

plt.suptitle('Sentinel-2 L2A — Washington D.C. area', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compute NDVI from Sentinel-2
ndvi_s2 = safe_index(s2_nir, s2_red)

# Save as GeoTIFF
s2_geobox = s2_scene.odc.geobox
save_single_band_geotiff(ndvi_s2, s2_geobox, DATA_DIR / 'sentinel2_ndvi.tif')

print(f'Sentinel-2 NDVI — mean: {np.nanmean(ndvi_s2):.3f}, std: {np.nanstd(ndvi_s2):.3f}')

---

## 6. Comparing Landsat vs Sentinel-2

A key skill for PhD-level remote sensing research is understanding when to choose one sensor over another. This section performs a direct comparison.

In [ ]:
# Side-by-side comparison of NDVI from both sensors
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

im0 = axes[0].imshow(ndvi_ls, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title(f'Landsat NDVI (30 m)\nmean={np.nanmean(ndvi_ls):.3f}', fontsize=12)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(ndvi_s2, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[1].set_title(f'Sentinel-2 NDVI (10 m)\nmean={np.nanmean(ndvi_s2):.3f}', fontsize=12)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('NDVI Comparison: Landsat vs Sentinel-2', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics comparison
import pandas as pd

stats = pd.DataFrame({
    'Sensor': ['Landsat 8/9', 'Sentinel-2'],
    'Resolution (m)': [30, 10],
    'NDVI Mean': [np.nanmean(ndvi_ls), np.nanmean(ndvi_s2)],
    'NDVI Std': [np.nanstd(ndvi_ls), np.nanstd(ndvi_s2)],
    'NDVI Min': [np.nanmin(ndvi_ls), np.nanmin(ndvi_s2)],
    'NDVI Max': [np.nanmax(ndvi_ls), np.nanmax(ndvi_s2)],
    'Pixel Count': [
        np.sum(~np.isnan(ndvi_ls)),
        np.sum(~np.isnan(ndvi_s2))
    ]
})

stats = stats.set_index('Sensor').round(4)
print(stats.to_string())
print('\nNote: Sentinel-2 has ~9x more pixels due to 10 m vs 30 m resolution.')

### ✏️ Exercise 6.1 — Sensor Comparison Discussion

Based on the comparison above, answer the following questions:

1. For mapping **individual trees in an urban park**, which sensor would you choose and why?
2. For monitoring **agricultural fields across a large region** (e.g., entire US Midwest), which sensor might be preferable?
3. Landsat has data going back to **1972**. How does temporal depth affect your sensor choice for a climate change study?
4. If you needed to detect a **small flood event** (~50 m wide river), which sensor would you use?

*Write your answers in the cell below as markdown.*

*Your answers here...*

---

## 7. Save Data Summary

Let's create a simple summary of what we've produced in this session.

In [ ]:
import json

summary = {
    'session': 'IGARSS26 Part 1',
    'study_area': {
        'name': 'Washington D.C. metro area',
        'bbox': AOI_BBOX
    },
    'landsat': {
        'collection': 'landsat-c2-l2',
        'scene_id': best_item.id,
        'date': str(best_item.datetime),
        'cloud_cover': best_item.properties.get('eo:cloud_cover'),
        'resolution_m': 30,
        'outputs': ['landsat_ndvi.tif', 'landsat_ndwi.tif', 'landsat_ndbi.tif']
    },
    'sentinel2': {
        'collection': 'sentinel-2-l2a',
        'scene_id': best_s2.id,
        'date': str(best_s2.datetime),
        'cloud_cover': best_s2.properties.get('eo:cloud_cover'),
        'resolution_m': 10,
        'outputs': ['sentinel2_ndvi.tif']
    }
}

summary_path = DATA_DIR / 'session1_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f'Summary saved to: {summary_path}')
print(json.dumps(summary, indent=2, default=str))

---

## ✅ Part 1 Summary

Excellent work! In this notebook you have:

- ✅ Connected to the Microsoft Planetary Computer STAC API
- ✅ Downloaded and processed **Landsat Collection 2 Level 2** surface reflectance data
- ✅ Downloaded and processed **Sentinel-2 Level 2A** surface reflectance data
- ✅ Computed **NDVI**, **NDWI**, and **NDBI** spectral indices
- ✅ Visualized true-colour and false-colour composites
- ✅ Saved analysis-ready **GeoTIFF** outputs
- ✅ Compared Landsat and Sentinel-2 outputs directly

### 🔜 Up next: Part 2

In **Part 2**, we will take these analysis-ready GeoTIFFs and build an **AI-assisted data processing pipeline** using LLM-based coding agents to automate multi-sensor land cover classification and change detection.

**Continue to `notebook_2_ai_agents.ipynb`** →